# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Lecture 9: how finite Markov chains move and settle

This notebook accompanies Sections 7.3--7.5. We use the row-vector
convention: after fixing the state order, $P(x,y)$ is the probability
of moving from $x$ to $y$, and $\mu_t=\mu_0P^t$.

The route is communication and period, hitting times, stationarity,
reversibility, graph walks, and a small PageRank model. All examples
are finite and simulations use a fixed seed.


In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(2026)
np.set_printoptions(precision=4, suppress=True)

def validate_transition_matrix(P, atol=1e-12):
    P = np.asarray(P, dtype=float)
    if P.ndim != 2 or P.shape[0] != P.shape[1]:
        raise ValueError("P must be square")
    if np.any(P < -atol) or not np.allclose(P.sum(axis=1), 1, atol=atol):
        raise ValueError("P must have nonnegative rows summing to one")
    return P

def simulate_paths(P, initial_state, n_steps, n_paths, rng):
    P = validate_transition_matrix(P)
    paths = np.empty((n_paths, n_steps + 1), dtype=int)
    paths[:, 0] = initial_state
    for t in range(n_steps):
        for state in range(len(P)):
            rows = paths[:, t] == state
            paths[rows, t + 1] = rng.choice(
                len(P), size=rows.sum(), p=P[state]
            )
    return paths


## Starting from a known distribution

The weather states are ordered as $(\mathsf D,\mathsf W)$. The
transition matrix alone is not a complete model; we also specify
$\mu_0=(1,0)$. The simulation uses a fresh random draw at each step.


In [ ]:
states = ("D", "W")
P_weather = validate_transition_matrix([
    [3 / 4, 1 / 4],
    [1 / 2, 1 / 2],
])
mu0 = np.array([1.0, 0.0])
times = np.arange(13)
exact = np.vstack([mu0 @ np.linalg.matrix_power(P_weather, t) for t in times])
paths = simulate_paths(P_weather, 0, 12, 30_000, rng)
empirical_wet = np.mean(paths == 1, axis=0)

print("state order:", states)
print("mu_0 P^2:", exact[2])
print("empirical/exact P(X_12=W):", empirical_wet[-1], exact[-1, 1])
plt.plot(times, exact[:, 1], "o-", label="exact")
plt.plot(times, empirical_wet, "x--", label="simulation")
plt.xlabel("time")
plt.ylabel("probability of wet")
plt.legend()
plt.show()


## Which states can reach each other?

State $y$ is accessible from $x$ if $P^n(x,y)>0$ for some
$n\geq0$. Two states communicate if each is accessible from the
other. A chain is irreducible when every pair communicates.

The return-time set is
$\mathcal T(x)=\{n\geq1:P^n(x,x)>0\}$; its period is the greatest
common divisor of this entire set. The finite-horizon calculation
below is only a diagnostic for these transparent examples.


In [ ]:
def transitive_closure(P):
    reachable = np.asarray(P) > 0
    np.fill_diagonal(reachable, True)
    for k in range(len(P)):
        reachable |= reachable[:, [k]] & reachable[[k], :]
    return reachable

def communication_classes(P):
    reachable = transitive_closure(P)
    unused, classes = set(range(len(P))), []
    while unused:
        x = min(unused)
        cls = {y for y in unused if reachable[x, y] and reachable[y, x]}
        classes.append(sorted(cls))
        unused -= cls
    return classes

def possible_return_times(P, state, horizon=12):
    power, times = np.eye(len(P)), []
    for n in range(1, horizon + 1):
        power = power @ P
        if power[state, state] > 1e-14:
            times.append(n)
    return times

P_alternating = validate_transition_matrix([[0, 1], [1, 0]])
for name, P in [("weather", P_weather), ("alternating", P_alternating)]:
    returns = possible_return_times(P, 0)
    print(name, communication_classes(P), returns, math.gcd(*returns))


The weather chain is irreducible and aperiodic because a one-step
return is possible. The alternating chain is irreducible with period
$2$: only even return times are possible.

## Chances and waiting times for reaching a state

For nonempty $A$, let $\tau_A=\inf\{t\geq0:X_t\in A\}$.
The hitting probability has boundary value $h_A=1$ on $A$ and
satisfies $h_A(x)=\sum_yP(x,y)h_A(y)$ outside $A$. If
$h_A(x)<1$, then $\mathbb E_x\tau_A=\infty$. On states where the
hit occurs with probability one, the finite mean satisfies
$m_A(x)=1+\sum_yP(x,y)m_A(y)$, with $m_A=0$ on $A$.


In [ ]:
def hitting_probabilities(P, targets):
    P = validate_transition_matrix(P)
    targets = np.array(sorted(set(targets)), dtype=int)
    can_reach = transitive_closure(P)[:, targets].any(axis=1)
    h = np.zeros(len(P))
    h[targets] = 1
    outside = ~np.isin(np.arange(len(P)), targets)
    unknown = np.flatnonzero(can_reach & outside)
    if len(unknown):
        Q = P[np.ix_(unknown, unknown)]
        r = P[np.ix_(unknown, targets)].sum(axis=1)
        h[unknown] = np.linalg.solve(np.eye(len(unknown)) - Q, r)
    return h

def mean_hitting_times(P, targets):
    targets = np.array(sorted(set(targets)), dtype=int)
    h = hitting_probabilities(P, targets)
    m = np.full(len(P), np.inf)
    m[targets] = 0
    outside = ~np.isin(np.arange(len(P)), targets)
    unknown = np.flatnonzero(np.isclose(h, 1, atol=1e-10) & outside)
    if len(unknown):
        Q = P[np.ix_(unknown, unknown)]
        m[unknown] = np.linalg.solve(np.eye(len(unknown)) - Q,
                                     np.ones(len(unknown)))
    return h, m

P_hit = validate_transition_matrix([
    [0.5, 0.4, 0.1],
    [0.0, 0.6, 0.4],
    [0.0, 0.0, 1.0],
])
print("certain hit:", mean_hitting_times(P_hit, [2]))

P_miss = validate_transition_matrix([
    [0.0, 0.5, 0.5],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
])
print("possible failure:", mean_hitting_times(P_miss, [1]))


In the second example, a chain from state $0$ hits state $1$ with
probability $1/2$. The hitting time is infinite with positive
probability, so its mean is infinite.

## A stationary distribution does not always mean convergence

A probability vector $\pi$ is stationary when $\pi P=\pi$. A
finite irreducible chain has one positive stationary distribution.
Convergence to it from every initial distribution additionally needs
aperiodicity.


In [ ]:
def unique_stationary_distribution(P):
    P = validate_transition_matrix(P)
    A = P.T - np.eye(len(P))
    b = np.zeros(len(P))
    A[-1], b[-1] = 1, 1
    pi = np.linalg.solve(A, b)
    if np.any(pi < -1e-10) or not np.allclose(pi @ P, pi):
        raise ValueError("computed vector is not stationary")
    pi = np.maximum(pi, 0)
    return pi / pi.sum()

pi_weather = unique_stationary_distribution(P_weather)
pi_alternating = unique_stationary_distribution(P_alternating)
print("weather stationary distribution:", pi_weather)
print("alternating stationary distribution:", pi_alternating)
mu = np.array([1.0, 0.0])
print(np.vstack([mu @ np.linalg.matrix_power(P_alternating, t)
                 for t in range(8)]))


The alternating laws oscillate although $(1/2,1/2)$ is stationary
and unique. This is exactly where aperiodicity enters the theorem.

## Reversible chains and walks on graphs

Reversibility means
$\pi(x)P(x,y)=\pi(y)P(y,x)$ for every pair. Summing over $x$
proves stationarity. For simple random walk on a finite connected
undirected graph, $\pi(v)$ is proportional to the vertex degree.


In [ ]:
vertices = np.arange(5)
edges = [(0, 1), (1, 2), (2, 0), (2, 3), (3, 4)]
adjacency = np.zeros((5, 5))
for v, w in edges:
    adjacency[v, w] = adjacency[w, v] = 1
degree = adjacency.sum(axis=1)
P_graph = adjacency / degree[:, None]
pi_graph = degree / degree.sum()
flow = pi_graph[:, None] * P_graph
print("classes:", communication_classes(P_graph))
print("pi:", pi_graph)
print("detailed-balance residual:", np.max(np.abs(flow - flow.T)))
print("stationarity residual:", pi_graph @ P_graph - pi_graph)

positions = np.array([[0, 1], [-0.9, -0.2], [0.9, -0.2],
                      [1.8, 0.3], [2.7, -0.2]])
for v, w in edges:
    plt.plot(positions[[v, w], 0], positions[[v, w], 1], "k-")
plt.scatter(positions[:, 0], positions[:, 1], s=800 * pi_graph + 80)
for v, (x, y) in enumerate(positions):
    plt.text(x, y, str(v), ha="center", va="center", color="white")
plt.axis("equal")
plt.axis("off")
plt.show()


## PageRank as a walk on a directed graph

Let $L$ follow outgoing links, using $q$ at a dangling page.
The random-surfer matrix
$P=(1-\alpha)L+\alpha\mathbf 1q$, with $0<\alpha<1$ and positive
$q$, has every entry positive. It is therefore irreducible and
aperiodic, and its unique stationary probabilities are PageRank scores.


In [ ]:
L = validate_transition_matrix([
    [0.0, 0.5, 0.5, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.5, 0.0, 0.0, 0.5],
    [0.25, 0.25, 0.25, 0.25],
])
alpha, q = 0.15, np.full(4, 0.25)
P_surfer = (1 - alpha) * L + alpha * np.tile(q, (4, 1))
print("PageRank:", unique_stationary_distribution(P_surfer))
print("all entries positive:", np.all(P_surfer > 0))


## Check your understanding

For the supplied chain, draw the graph; find the communication
classes and period; calculate the probability and mean time to hit
state $2$; and verify a stationary distribution. The complete code
gives numerical checks, but the exercise is to justify them from the
definitions and first-step equations.


In [ ]:
P_task = validate_transition_matrix([
    [0.2, 0.8, 0.0],
    [0.1, 0.5, 0.4],
    [0.3, 0.0, 0.7],
])
print("classes:", communication_classes(P_task))
print("returns:", possible_return_times(P_task, 0))
print("hit state 2:", mean_hitting_times(P_task, [2]))
print("stationary:", unique_stationary_distribution(P_task))
